#  PageIndex — Vectorless RAG Crash Course

### 🔑 Key Concept
 - Traditional RAG → chunk → embed → cosine similarity → retrieve
 - PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

The problem with vector RAG:
 - Similarity ≠ Relevance
 - A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

In [3]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if GROQ_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:    ✅


In [5]:
from pageindex import PageIndexClient
from groq import Groq

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


### Section 2: Upload & Index a PDF
What happens here:

 - Upload your PDF to the PageIndex cloud
 - PageIndex uses an LLM to read the document structure
 - Builds a hierarchical tree index (like a smart Table of Contents)
 - Returns a doc_id for all future operations
 - Why NO chunking?
 - Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [6]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "./sample_document.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: ./sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmurrzh27001s0dp178rhr569
   (Save this ID — you'll use it throughout the notebook)


In [7]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


### 🔍 Section 3: Inspect the Tree Structure

In [8]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 1

🌲 Raw tree (first node):
{
  "title": "Which kind of research papers influence policymaking",
  "node_id": "0000",
  "start_index": 1,
  "end_index": 30,
  "summary": "This section explores the determinants of research paper influence on policymaking by analyzing a dataset of nearly 150,000 articles. It reviews existing literature on science-policy intersections and altmetrics, outlines a regression-based methodology incorporating journal- and article-level metrics, and presents results showing that traditional scholarly citations, high journal impact, Mendeley readership, and news or blog mentions positively correlate with policy citations, whereas X (social media) mentions have a negative effect. Finally, the discussion evaluates these findings, addresses methodological limitations, and considers the implications for bridging scientific knowledge and public policy.",
  "text": "# Which kind of research papers influence policymaking\n\nPablo Dorta-Gonz\u00e1le

In [9]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Which kind of research papers influence policymaking  (p.?)
  └─ [0001] Abstract  (p.?)
  └─ [0002] 1. Introduction  (p.?)
  └─ [0003] 2. Literature review  (p.?)
  └─ [0004] 3. Methodology  (p.?)
    └─ [0005] *Objectives of the study*  (p.?)
    └─ [0006] *Data*  (p.?)
    └─ [0007] *Journal-level metrics*  (p.?)
    └─ [0008] *Article-level metrics*  (p.?)
    └─ [0009] *Methods*  (p.?)
  └─ [0010] 4. Results  (p.?)
  └─ [0011] 5. Discussion  (p.?)
    └─ [0012] 5. Discussion (intro)  (p.?)
    └─ [0013] *Strong effect: Scientific references*  (p.?)
    └─ [0014] *Moderate impact: Journal influence and Mendeley readership*  (p.?)
    └─ [0015] *Minor impact: News coverage, Blog engagement, and hybrid accessibility*  (p.?)
    └─ [0016] *Minor adverse impact: Social media references*  (p.?)
    └─ [0017] *Negligible adverse impact: Journal reputation and expert evaluations*  (p.?)
    └─ [0018] *Apparent contradiction in the sign of effects at the j

In [10]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")

🔢 Total nodes in tree: 22


### 🧠 Section 4: LLM Tree Search — The Core of PageIndex
 - This is where PageIndex fundamentally differs from vector RAG.

1. Vector RAG retrieval:
 - query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
 - Problem: finds what's similar, not what's relevant

2. PageIndex retrieval:
 - query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
 - Advantage: LLM understands document structure, context, and intent

- The LLM acts like a human expert scanning a Table of Contents.

In [11]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "openai/gpt-oss-20b") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [12]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the methodology of the paper?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the methodology of the paper?

🧠 LLM Reasoning:
The query asks for the methodology of the paper. The document tree clearly labels section 0004 as "3. Methodology" and within it, section 0009 titled "*Methods*" provides the detailed methodological approach. Therefore, the most relevant node IDs are 0004 for the overall methodology section and 0009 for the specific methods details.

🎯 Selected Node IDs: ['0004', '0009']


### ⚙️ Section 5: Full End-to-End RAG Pipeline
3 steps:

 - Tree Search → LLM picks relevant node_ids
 - Retrieve → Fetch the actual section content from those nodes
 - Generate → LLM writes a grounded answer with page citations
 - What makes this better than vector RAG:

 - Retrieved content has titles + page numbers (traceable)
 - LLM can cite exactly which section the answer comes from
 - No hallucination from irrelevant chunks

In [13]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [17]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-20b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [18]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [19]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the methodology covered in the paper?",
    tree=pageindex_tree
)

🔍 Query: What are the methodology covered in the paper?

🧠 Reasoning: The question asks for the methodology covered in the paper. The main methodology section is node 0004, and its children (0005: Objectives, 0006: Data, 0007: Journal-level metrics, 0008: Article-level ...
🎯 Retrieved node IDs: ['0004', '0005', '0006', '0007', '0008', '0009']
📄 Sections found: ['3. Methodology', '*Objectives of the study*', '*Data*', '*Journal-level metrics*', '*Article-level metrics*', '*Methods*']

📝 Answer:
**Methodologies used in the paper**

1. **Dataset construction and harmonisation** – Systematic retrieval of journal titles from the APSA list, matching with Scopus to obtain journal‑level metrics (ISSN, SJR, quartile, citations per document), and cross‑referencing with Altmetric.com for article‑level metrics, producing a final sample of 149 557 policy‑relevant journal articles (Section *Data*, Page 5).  

2. **Use of expert and bibliometric indicators** – Inclusion of APSA journal ratings, Scima